In [ ]:
import os
import re
import subprocess
import shutil
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# set some naming
fds_template_file = "pool.fds"
slurm_template = "jureca.template"
global_mesh = "&MESH IJK=64,64,128, XB=-2.5,2.5,-2.5,2.5,0.0,10 /"
decomposition_script = "decompose_fds_mesh.py"
root_dir = "rundir"
dir_prefix = "mesh_"

In [ ]:
# target decompositions
mesh_decomposition = {
    1: "1,1,1",
    2: "1,1,2",
    4: "2,1,2",
    8: "2,2,2",
    16: "2,2,4",
    32: "4,2,4",
    64: "4,4,4",
    128: "4,4,8",
    256: "8,4,8"
}

In [ ]:
os.mkdir(root_dir)
for nm in mesh_decomposition:
    print("Processing mesh decomposition:", nm, "->", mesh_decomposition[nm])
    
    # setup run directory
    target_dir = os.path.join(root_dir,f"{dir_prefix}{nm:04d}")
    os.mkdir(target_dir)

    # create mesh input file
    mf = open(os.path.join(target_dir, 'mesh.input'), 'w')
    res = subprocess.call(f"python3 {decomposition_script} \"{global_mesh}\" \"{mesh_decomposition[nm]}\"" , shell=True, stdout=mf)
    mf.close()

    # copy fds inputfile with mesh include
    shutil.copyfile(fds_template_file, os.path.join(target_dir, fds_template_file))

    # copy and adjust slurm job file
    sif = open(slurm_template, 'r')
    sof = open(os.path.join(target_dir, 'jureca.job'), 'w')
    slurm_in = sif.read()
    sif.close()
    slurm_out = slurm_in.replace("#NTASKS#", str(nm))
    sof.write(slurm_out)
    sof.close()

In [ ]:
# find all cpu files
cpu_files = sorted(glob.glob('rundir/*/*_cpu.csv'))
print(cpu_files)

In [ ]:
lmeshes = []
lwct = []
for cf in cpu_files:
    print("Processing CPU file:", cf)
    nm = int(re.search('mesh_\d+', cf)[0].split('_')[-1])
    print("Number of meshes:", nm)

    data = pd.read_csv(cf)
    # print(data)
    lmeshes.append(nm)
    lwct.append(data["Total T_USED (s)"][0])

meshes = np.array(lmeshes)
wct = np.array(lwct)

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(np.log2(meshes), wct, '-o')

plt.xticks([0, 1, 2, 3, 4, 5, 6, 7, 8], [1, 2, 4, 8, 16, 32, 64, 128, 256])
plt.yscale('log')
plt.grid()
plt.xlabel('number of meshes / cores')
plt.ylabel('total time used / s')
plt.savefig('benchmark_time.png', dpi=300)

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(meshes, 1/(wct/wct[0]), '-o')
plt.plot([1,20], [1,20], color='grey')

plt.grid()
plt.xlabel('number of meshes / cores')
plt.ylabel('speedup')
plt.savefig('benchmark_speedup.png', dpi=300)